# Bell Nonlocality — From NPA Bases to Optimised Relaxations

This notebook walks through the complete **Bell** pipeline:

| Step | What you'll see |
|------|----------------|
| **1** | Setup & imports |
| **2** | Bell scenarios and projector-word algebra |
| **3** | NPA basis generation |
| **4** | Moment-matrix compilation & structure |
| **5** | Bell operators (CHSH, custom) |
| **6** | Solving an SDP relaxation (→ Tsirelson's bound) |
| **7** | Inspecting the solution |
| **8** | Single optimisation run (basis selection) |
| **9** | Sweeping over *k* values and seeds |
| **10** | Atomic persistence with a callback |
| **11** | Loading and discovering saved results |
| **12** | CLI reference |

---
## 1. Setup & Imports

In [1]:
import sys, json, shutil
import numpy as np
from pathlib import Path

# Add the project root so top-level packages are importable
PROJECT_ROOT = str(Path("..").resolve())
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

# --- Bell algebra layer ---
from bell.bell_logic import (
    BellScenario,
    BellWord,
    IDENTITY,
    alice_projector,
    bob_projector,
    multiply_words,
    generate_npa_basis,
    compile_moment_matrix_rep,
    BellOperator,
    Sense,
    expand_bell_operator,
    chsh_operator,
    probability_word,
    basis_summary,
    rep_summary,
)

# --- SDP layer ---
from bell.bell_sdp import (
    BellMomentSDP,
    build_bell_sdp,
    build_moment_matrix_expression,
    compile_operator_linear_form,
)

# --- Optimisation layer ---
from bell.bell_optimize import (
    build_bell_basis_sets,
    BellOptimizationResult,
    OnResultCallback,
    optimize_bell_relaxation,
    sweep_k_values,
)

# --- Persistence ---
from artifact_manager import ArtifactManager, RunDir, config_hash

print("All imports OK ✓")
print(f"Project root: {PROJECT_ROOT}")

All imports OK ✓
Project root: /Users/cisco/_Code/MyRepositories/SpinsSDP


---
## 2. Bell Scenario & Word Algebra

A **Bell scenario** is defined by:
- $m_A, m_B$: number of measurement settings for Alice and Bob
- $d_A, d_B$: number of outcomes per setting

For CHSH we use the simplest nontrivial case: $m_A = m_B = 2$, $d_A = d_B = 2$.

**Words** represent products of projectors. A `BellWord` is a pair of sequences: Alice's projector labels and Bob's. The NPA hierarchy expresses quantum correlations through the *moment matrix* $\Gamma_{ij} = \langle \psi | S_i^\dagger S_j | \psi \rangle$ where each $S_i$ is a word.

In [2]:
# --- Define the CHSH scenario ---
scenario = BellScenario.symmetric(m=2, d=2)
print(f"Scenario: m_A={scenario.m_A}, m_B={scenario.m_B}, "
      f"d_A={scenario.d_A}, d_B={scenario.d_B}")

# --- Projector words ---
a00 = alice_projector(scenario, x=0, a=0)
a10 = alice_projector(scenario, x=1, a=0)
b00 = bob_projector(scenario, y=0, b=0)
b10 = bob_projector(scenario, y=1, b=0)

print(f"\nAlice projectors:  A(x=0,a=0) = {a00},  A(x=1,a=0) = {a10}")
print(f"Bob projectors:    B(y=0,b=0) = {b00},  B(y=1,b=0) = {b10}")
print(f"Identity:          {IDENTITY}")

# --- Word multiplication ---
# Alice and Bob commute, so A·B = the joint product
ab = multiply_words(a00, b00)
print(f"\nA(0,0) · B(0,0) = {ab}")

# Same-party product: A(0,0)·A(0,0) = A(0,0)  (projectors are idempotent)
aa = multiply_words(a00, a00)
print(f"A(0,0) · A(0,0) = {aa}")

# Orthogonal projectors: A(0,0)·A(0,1) = 0 (returns None)
a01 = alice_projector(scenario, x=0, a=1)
zero = multiply_words(a00, a01)
print(f"A(0,0) · A(0,1) = {zero}  (None means zero)")

Scenario: m_A=2, m_B=2, d_A=2, d_B=2

Alice projectors:  A(x=0,a=0) = E_0|0,  A(x=1,a=0) = E_0|1
Bob projectors:    B(y=0,b=0) = F_0|0,  B(y=1,b=0) = F_0|1
Identity:          I

A(0,0) · B(0,0) = E_0|0 F_0|0
A(0,0) · A(0,0) = E_0|0
A(0,0) · A(0,1) = None  (None means zero)


---
## 3. NPA Basis Generation

The NPA hierarchy at level $k$ builds a set of **words** of length up to $k$
from the scenario's projectors. Level 0 contains only the identity $\mathbb{1}$,
level 1 adds all single projectors, level 2 adds all products of two projectors, etc.

`generate_npa_basis(scenario, k)` returns a `BellBasis` object with:
- `.words` — all words (sorted)
- `.levels` — words grouped by level
- `.index` — word → integer index
- `.min_len` — word → minimum representation length

In [6]:
# Generate NPA bases at levels 1 and 2
basis_k1 = generate_npa_basis(scenario, k=1)
basis_k2 = generate_npa_basis(scenario, k=2)
print(basis_k1)
print("=== NPA Level 1 ===")
print(basis_summary(basis_k1))

print()

print("=== NPA Level 2 ===")
print(basis_summary(basis_k2))


BellBasis(scenario=BellScenario(m_A=2, m_B=2, d_A=2, d_B=2), k=1, words=[I, F_0|0, F_0|1, E_0|0, E_0|1], levels=[[I], [F_0|0, F_0|1, E_0|0, E_0|1]], min_len={I: 0, E_0|0: 1, E_0|1: 1, F_0|0: 1, F_0|1: 1}, index={I: 0, F_0|0: 1, F_0|1: 2, E_0|0: 3, E_0|1: 4})
=== NPA Level 1 ===
Bell NPA Basis for scenario (2, 2) with d_A=2, d_B=2
Level k = 1
Total words: 5
  Level 0: 1 words
  Level 1: 4 words

=== NPA Level 2 ===
Bell NPA Basis for scenario (2, 2) with d_A=2, d_B=2
Level k = 2
Total words: 13
  Level 0: 1 words
  Level 1: 4 words
  Level 2: 8 words


In [27]:
# List all words in the level-1 basis
print("Level-1 words:")
for i, w in enumerate(basis_k1.words):
    print(f"  [{i}] {w}")

print()

# Level breakdown
print("Words per level:")
for lvl_idx, lvl_words in enumerate(basis_k1.levels):
    names = [str(w) for w in lvl_words]
    print(f"  Level {lvl_idx}: {len(lvl_words)} words → {names}")

Level-1 words:
  [0] I
  [1] F_0|0
  [2] F_0|1
  [3] E_0|0
  [4] E_0|1

Words per level:
  Level 0: 1 words → ['I']
  Level 1: 4 words → ['F_0|0', 'F_0|1', 'E_0|0', 'E_0|1']


---
## 4. Moment Matrix Compilation & Structure

`compile_moment_matrix_rep(basis)` takes a `BellBasis` and computes the
**moment matrix representation**: it figures out, for every pair $(i, j)$
of basis words, what $S_i^\dagger S_j$ reduces to after applying the
projector algebra (idempotence, orthogonality, commutativity of Alice/Bob).

The result is a `BellMomentMatrixRep` with:
- `.labels` — unique moment labels (one per independent entry)
- `.label_index` — maps `(i, j)` → label index
- `.idx_I` — index of the identity label (the $(0,0)$ entry)

In [6]:
# Compile the moment matrix representation for level-1
rep_k1 = compile_moment_matrix_rep(basis_k1)
rep_summary(rep_k1)

print(f"\nMoment matrix size: {len(basis_k1.words)} × {len(basis_k1.words)}")
print(f"Number of unique labels (free variables): {len(rep_k1.labels)}")
print(f"Identity label index: {rep_k1.idx_I}")


Moment matrix size: 5 × 5
Number of unique labels (free variables): 11
Identity label index: 0


In [7]:
# Print the symbolic moment matrix: which label appears in each cell
n = len(basis_k1.words)
word_names = [str(w) for w in basis_k1.words]

print("Symbolic moment matrix  Γ[i,j] = label_index  (for level-1 basis):\n")
header = "        " + "  ".join(f"{w:>8s}" for w in word_names)
print(header)
print("        " + "-" * (10 * n))
for i in range(n):
    row_str = f"{word_names[i]:>8s}|"
    for j in range(n):
        label_idx = rep_k1.label_idx[i, j]
        if label_idx < 0:
            row_str += f"{'  zero':>10s}"
        else:
            row_str += f"{label_idx:>10d}"
    print(row_str)

Symbolic moment matrix  Γ[i,j] = label_index  (for level-1 basis):

               I     F_0|0     F_0|1     E_0|0     E_0|1
        --------------------------------------------------
       I|         0         1         2         3         4
   F_0|0|         1         1         5         6         9
   F_0|1|         2         5         2         7        10
   E_0|0|         3         6         7         3         8
   E_0|1|         4         9        10         8         4


---
## 5. Bell Operators

A **Bell operator** is a `Dict[BellWord, float]` mapping words to coefficients.
The library provides:
- `chsh_operator(scenario)` — the standard CHSH operator
- `expand_bell_operator(op, scenario)` — applies the completeness relation
  $\sum_a \Pi^A_{x,a} = \mathbb{1}$ to express the operator in terms of
  individual projectors (needed for the SDP)
- `probability_word(scenario, x, a, y, b)` — the word $\Pi^A_{x,a} \Pi^B_{y,b}$
  representing the joint probability $P(a,b|x,y)$

In [8]:
# --- The CHSH operator ---
chsh_op = chsh_operator(scenario)
print("CHSH operator (compact form):")
for word, coeff in sorted(chsh_op.items(), key=lambda t: str(t[0])):
    print(f"  {coeff:+.1f} · {word}")

# --- Expand via completeness substitution ---
chsh_expanded = expand_bell_operator(chsh_op, scenario)
print(f"\nExpanded CHSH operator ({len(chsh_expanded)} terms):")
for word, coeff in sorted(chsh_expanded.items(), key=lambda t: str(t[0])):
    if abs(coeff) > 1e-12:
        print(f"  {coeff:+.1f} · {word}")

CHSH operator (compact form):
  -4.0 · E_0|0
  +4.0 · E_0|0 F_0|0
  +4.0 · E_0|0 F_0|1
  +0.0 · E_0|1
  +4.0 · E_0|1 F_0|0
  -4.0 · E_0|1 F_0|1
  -4.0 · F_0|0
  +0.0 · F_0|1
  +2.0 · I

Expanded CHSH operator (7 terms):
  -4.0 · E_0|0
  +4.0 · E_0|0 F_0|0
  +4.0 · E_0|0 F_0|1
  +4.0 · E_0|1 F_0|0
  -4.0 · E_0|1 F_0|1
  -4.0 · F_0|0
  +2.0 · I


In [28]:
# --- Probability words ---
# P(a=0, b=0 | x=0, y=0)
p_word = probability_word(scenario, x=0, a=0, y=0, b=0)
print(f"P(a=0, b=0 | x=0, y=0)  →  word: {p_word}")

# --- Build a custom operator from probability words ---
# Example: P(a=b | x=0, y=0) = P(0,0|0,0) + P(1,1|0,0)
custom_op: BellOperator = {
    probability_word(scenario, 0, 0, 0, 0): 1.0,
    probability_word(scenario, 0, 1, 0, 1): 1.0,
}
print("\nCustom operator  P(a=b | x=0, y=0):")
for word, coeff in custom_op.items():
    print(f"  {coeff:+.1f} · {word}")

P(a=0, b=0 | x=0, y=0)  →  word: E_0|0 F_0|0

Custom operator  P(a=b | x=0, y=0):
  +1.0 · E_0|0 F_0|0
  +1.0 · E_1|0 F_1|0


---
## 6. Solving an SDP Relaxation

`build_bell_sdp(rep, objective_op, sense="max")` assembles the full CVXPY problem:
- Moment matrix $\Gamma \succeq 0$
- $\Gamma_{0,0} = 1$ (normalisation)
- Objective: maximise (or minimise) $\sum_\ell c_\ell \, y_\ell$

For CHSH at NPA level 1, the SDP is already tight and recovers **Tsirelson's bound** $2\sqrt{2} \approx 2.828$.

In [99]:
import time

# Build the SDP for CHSH at NPA level 1
sdp_k1 = build_bell_sdp(rep_k1, chsh_expanded, sense="max")

print(f"SDP problem built.")
print(f"  Moment matrix size: {sdp_k1.M.shape}")
print(f"  Number of moment variables: {sdp_k1.y.size}")
print(f"  Number of constraints: {len(sdp_k1.constraints)}")

# Solve with MOSEK
t0 = time.perf_counter()
sdp_k1.problem.solve(solver="MOSEK", verbose=False)
dt = time.perf_counter() - t0

tsirelson = 2 * np.sqrt(2)

print(f"\n{'='*50}")
print(f"  SDP status:     {sdp_k1.problem.status}")
print(f"  SDP value:      {sdp_k1.problem.value:.6f}")
print(f"  Tsirelson bound: {tsirelson:.6f}")
print(f"  Gap:            {abs(sdp_k1.problem.value - tsirelson):.2e}")
print(f"  Solve time:     {dt:.3f}s")
print(f"{'='*50}")

SDP problem built.
  Moment matrix size: (5, 5)
  Number of moment variables: 11
  Number of constraints: 2

  SDP status:     optimal
  SDP value:      2.828427
  Tsirelson bound: 2.828427
  Gap:            6.28e-10
  Solve time:     0.004s


---
## 7. Inspecting the Solution

After solving we can extract:
1. The **optimal moment matrix** $\Gamma^*$ and verify it is PSD
2. The **moment variable values** $y_\ell^*$
3. How the bound changes with the NPA level

In [32]:
# --- Optimal moment matrix ---
Gamma = sdp_k1.M.value
print("Optimal moment matrix Γ*:")
np.set_printoptions(precision=4, suppress=True, linewidth=120)
print(Gamma)

# --- PSD check ---
eigenvalues = np.linalg.eigvalsh(Gamma)
print(f"\nEigenvalues of Γ*: {eigenvalues}")
print(f"All ≥ 0?  {'Yes ✓' if np.all(eigenvalues > -1e-8) else 'No ✗'}")

# --- Moment variable values ---
y_vals = sdp_k1.y.value
print(f"\nMoment variable values (y):")
for i, val in enumerate(y_vals):
    label = rep_k1.labels[i]
    print(f"  y[{i}] = {val:+.6f}   (label: {label})")

Optimal moment matrix Γ*:
[[1.     0.4191 0.4655 0.4184 0.4671]
 [0.4191 0.4191 0.1923 0.3455 0.3699]
 [0.4655 0.1923 0.4655 0.3687 0.0396]
 [0.4184 0.3455 0.3687 0.4184 0.1928]
 [0.4671 0.3699 0.0396 0.1928 0.4671]]

Eigenvalues of Γ*: [-0.      0.      0.279   0.5     1.9911]
All ≥ 0?  Yes ✓

Moment variable values (y):
  y[0] = +1.000000   (label: I)
  y[1] = +0.419055   (label: F_0|0)
  y[2] = +0.465532   (label: F_0|1)
  y[3] = +0.418390   (label: E_0|0)
  y[4] = +0.467138   (label: E_0|1)
  y[5] = +0.192293   (label: F_0|0 F_0|1)
  y[6] = +0.345496   (label: E_0|0 F_0|0)
  y[7] = +0.368740   (label: E_0|0 F_0|1)
  y[8] = +0.192763   (label: E_0|0 E_0|1)
  y[9] = +0.369876   (label: E_0|1 F_0|0)
  y[10] = +0.039560   (label: E_0|1 F_0|1)


In [ ]:
# --- Compare bounds across NPA levels ---
print("Bound vs. NPA level:\n")
for k in [1, 2]:
    basis = generate_npa_basis(scenario, k)
    rep = compile_moment_matrix_rep(basis)
    sdp = build_bell_sdp(rep, chsh_expanded, sense="max")
    sdp.problem.solve(solver="MOSEK", verbose=False)
    val = sdp.problem.value
    gap = abs(val - tsirelson)
    print(f"  k={k}:  basis size = {len(basis.words):>3d},  "
          f"bound = {val:.6f},  gap to 2√2 = {gap:.2e}")

Bound vs. NPA level:

  k=1:  basis size =   5,  bound = 2.828427,  gap to 2√2 = 6.28e-10
  k=2:  basis size =  13,  bound = 2.828427,  gap to 2√2 = 1.70e-08


---
## 8. Single Optimisation Run (Basis Selection)

The full NPA level-2 basis may contain many words. The **optimisation layer**
asks: *can we get a tight bound using only $k$ extra words from level 2,
added to the full level-1 basis?*

`optimize_bell_relaxation()` is the main entry point. It:
1. Builds the **starting set** (level-1 words) and **adding set** (level-2 words not in level 1).
2. Creates an objective function that, for a given selection of $k$ words,
   compiles the SDP and solves it.
3. Uses a chosen optimiser (SA, PT, BO, RBM, or random) to find the best $k$ words.

In [34]:
# Look at the basis sets first
starting_set, adding_set, final_set = build_bell_basis_sets(scenario, start_level=1, end_level=2)

print(f"Starting set (level ≤ 1): {len(starting_set)} words")
print(f"Adding set   (level 2):   {len(adding_set)} words")
print(f"Final set    (all):       {len(final_set)} words")

print("\nAdding set words:")
for i, w in enumerate(adding_set):
    print(f"  [{i}] {w}")

Starting set (level ≤ 1): 5 words
Adding set   (level 2):   8 words
Final set    (all):       13 words

Adding set words:
  [0] F_0|0 F_0|1
  [1] F_0|1 F_0|0
  [2] E_0|0 F_0|0
  [3] E_0|0 F_0|1
  [4] E_0|0 E_0|1
  [5] E_0|1 F_0|0
  [6] E_0|1 F_0|1
  [7] E_0|1 E_0|0


In [105]:
# Run a single optimisation: pick k=2 words from the adding set
result = optimize_bell_relaxation(
    scenario=scenario,
    bell_operator=chsh_op,       # un-expanded is fine — expanded internally
    start_level=1,
    end_level=2,
    k=2,
    method="sa",
    seed=42,
    verbose=True,
    mosek_tol=10e-12
)

print(f"\n--- Result ---")
print(f"  Best SDP value:  {result.best_value:.6f}")
print(f"  Selected indices: {result.best_indices}")
print(f"  Selected words:   {[str(adding_set[i]) for i in result.best_indices]}")
print(f"  Time:            {result.elapsed_s:.3f}s")
print(f"  Method:          {result.method}")
print(f"  k:               {result.k}")
print(f"  Seed:            {result.seed}")

Bell optimisation: scenario=(2,2), d=(2,2)
  Starting set: 5 words, Adding set: 8 words, k=2
  Method: sa, sense=max
  Result: SDP value = 2.828427, time = 0.40s, evals = 101

--- Result ---
  Best SDP value:  2.828427
  Selected indices: [0, 7]
  Selected words:   ['F_0|0 F_0|1', 'E_0|1 E_0|0']
  Time:            0.402s
  Method:          sa
  k:               2
  Seed:            42


---
## 9. Sweeping over *k* Values and Seeds

`sweep_k_values()` runs the optimisation for multiple $(k, \text{seed})$ pairs.

Key features:
- **`feedback=True`**: chains results across $k$ values — the best mask at $k_i$ is
  passed as warm-start to $k_{i+1}$ (per seed).
- **`existing_results`**: skip already-computed jobs (for resume support).
- **`on_result`**: callback invoked after each new result (for atomic persistence).
- **`seeds`**: run multiple random seeds per $k$ for statistical robustness.

The function returns a dict mapping `(k, seed) → BellOptimizationResult`.

In [104]:
# Sweep over k=1,2,3 with 2 seeds each
results = sweep_k_values(
    scenario=scenario,
    bell_operator=chsh_op,
    start_level=1,
    end_level=2,
    k_values=[0, 1, 2, 3],
    seeds=[42, 43],
    method="sa",
    verbose=True,
    mosek_tol=10e-12
)

# Inspect the results
print(f"\nAll (k, seed) pairs computed: {len(results)}")
print(f"\n{'k':>3s}  {'seed':>4s}  {'SDP value':>10s}  {'time (s)':>8s}  {'evals':>5s}")
print("-" * 40)
for (k, seed), res in sorted(results.items()):
    print(f"{k:>3d}  {seed:>4d}  {res.best_value:>10.6f}  {res.elapsed_s:>8.3f}  {res.n_obj_evals:>5d}")

Bell optimisation sweep: 100%|██████████| 8/8 [00:02<00:00,  3.37it/s, k=3, seed=43] 


Best SDP value per k:
  k=0: 2.828427
  k=1: 2.828427
  k=2: 2.828427
  k=3: 2.828427

All (k, seed) pairs computed: 8

  k  seed   SDP value  time (s)  evals
----------------------------------------
  0    42    2.828427     0.008    101
  0    43    2.828427     0.005    101
  1    42    2.828427     0.370    101
  1    43    2.828427     0.371    101
  2    42    2.828427     0.387    101
  2    43    2.828427     0.389    101
  3    42    2.828427     0.420    101
  3    43    2.828427     0.416    101


In [106]:
# --- Resume support: existing_results ---
# Pass previous results to skip already-computed jobs.
# Here we re-run the same sweep but pass the results we already have:

results_resumed = sweep_k_values(
    scenario=scenario,
    bell_operator=chsh_op,
    start_level=1,
    end_level=2,
    k_values=[1, 2, 3, 4, 5, 6],       # added k=4,5,6
    seeds=[42, 43],
    method="sa",
    existing_results=results,     # ← skip (k=1..3, seed=42,43) — already done
    verbose=True,
    mosek_tol=10e-12
)

# Only k=4 was computed; k=1,2,3 were skipped
print(f"\nTotal results after resume: {len(results_resumed)}")

Bell optimisation sweep: 100%|██████████| 6/6 [00:02<00:00,  2.15it/s, k=6, seed=43]


Best SDP value per k:
  k=1: 2.828427
  k=2: 2.828427
  k=3: 2.828427
  k=4: 2.828427
  k=5: 2.828427
  k=6: 2.828427

Total results after resume: 14


---
## 10. Atomic Persistence with a Callback

The `on_result` callback is called after **every newly completed** `(k, seed)` job.
This lets you save results **atomically** so a crash loses at most the current computation.

The pattern:
1. Create a `RunDir` via `ArtifactManager.create_run()`.
2. Define a callback that **appends** the new result to an accumulator and calls
   `run.save_table(...)` — which writes atomically (temp file → `os.replace`).
3. Pass the callback as `on_result=` to `sweep_k_values()`.

Let's wire this up end-to-end.

In [107]:
# --- Set up a clean demo directory ---
demo_root = Path("bell_demo_results")
if demo_root.exists():
    shutil.rmtree(demo_root)
demo_root.mkdir()

am = ArtifactManager(demo_root)

# --- Create a run ---
sweep_config = {
    "scenario": {"m_A": 2, "m_B": 2, "d_A": 2, "d_B": 2},
    "operator": "chsh",
    "start_level": 1,
    "end_level": 2,
    "method": "random",
}

run = am.create_run(
    artifact="bell_optimization_sweep",
    name="chsh_lv1to2_random_demo",
    config=sweep_config,
)
print(f"Run directory: {run.path}")
print(f"Config hash:   {config_hash(sweep_config)}")

Run directory: bell_demo_results/bell_optimization_sweep/v1/chsh_lv1to2_random_demo
Config hash:   7b604512ecface13


In [108]:
# --- Define the atomic callback ---

# Accumulators: these grow as results come in
_acc_k = []
_acc_seed = []
_acc_best_value = []
_acc_elapsed_s = []
_acc_n_obj_evals = []
_acc_mask_bits = []

def on_result(k: int, seed: int, res: BellOptimizationResult) -> None:
    """Save every new result atomically via RunDir.save_table()."""
    packed_mask = np.packbits(res.mask.astype(np.uint8))

    _acc_k.append(k)
    _acc_seed.append(seed)
    _acc_best_value.append(res.best_value)
    _acc_elapsed_s.append(res.elapsed_s)
    _acc_n_obj_evals.append(res.n_obj_evals)
    _acc_mask_bits.append(packed_mask)

    # Atomic save — the full table is rewritten each time
    run.save_table(
        k=np.array(_acc_k, dtype=np.int32),
        seed=np.array(_acc_seed, dtype=np.int32),
        best_value=np.array(_acc_best_value, dtype=np.float64),
        elapsed_s=np.array(_acc_elapsed_s, dtype=np.float64),
        n_obj_evals=np.array(_acc_n_obj_evals, dtype=np.int32),
        mask_bits=np.stack(_acc_mask_bits, axis=0),
    )
    print(f"    💾 Checkpoint: saved {len(_acc_k)} result(s) to disk")

print("Callback defined ✓")

Callback defined ✓


In [109]:
# --- Run the sweep with atomic persistence ---
sweep_results = sweep_k_values(
    scenario=scenario,
    bell_operator=chsh_op,
    start_level=1,
    end_level=2,
    k_values=[1, 2, 3],
    seeds=[42, 43],
    method="random",
    on_result=on_result,           # ← atomic save after each job
    verbose=True,
)

# Finalise metadata
run.update_meta(
    status="complete",
    k_values=[1, 2, 3],
    seeds=[42, 43],
    total_runs=len(_acc_k),
)

print(f"\nSweep complete — {len(_acc_k)} results saved to {run.path}")

Bell optimisation sweep: 100%|██████████| 6/6 [00:00<00:00, 102.25it/s, k=3, seed=43]

    💾 Checkpoint: saved 1 result(s) to disk
    💾 Checkpoint: saved 2 result(s) to disk
    💾 Checkpoint: saved 3 result(s) to disk
    💾 Checkpoint: saved 4 result(s) to disk
    💾 Checkpoint: saved 5 result(s) to disk
    💾 Checkpoint: saved 6 result(s) to disk

Best SDP value per k:
  k=1: 2.828427
  k=2: 2.828428
  k=3: 2.828428

Sweep complete — 6 results saved to bell_demo_results/bell_optimization_sweep/v1/chsh_lv1to2_random_demo


In [111]:
# --- What's on disk? ---
import os

print("On-disk layout:\n")
for dirpath, dirnames, filenames in os.walk(demo_root):
    level = dirpath.replace(str(demo_root), "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(dirpath)}/")
    sub_indent = "  " * (level + 1)
    for f in filenames:
        fpath = Path(dirpath) / f
        size = fpath.stat().st_size
        print(f"{sub_indent}{f}  ({size} bytes)")

On-disk layout:

bell_demo_results/
  bell_optimization_sweep/
    v1/
      chsh_lv1to2_random_demo/
        meta.json  (470 bytes)
        data.npz  (1676 bytes)


Now let's pretend we're coming back later and want to load the results.

In [112]:
# --- Re-open and list runs ---
am2 = ArtifactManager(demo_root)

print("Available artifacts:")
for artifact_dir in sorted(am2.root.iterdir()):
    if artifact_dir.is_dir():
        runs = am2.list_runs(artifact_dir.name)
        print(f"  {artifact_dir.name}: {len(runs)} run(s)")

# --- Load the table ---
loaded_run = am2.open_run(
    artifact="bell_optimization_sweep",
    name="chsh_lv1to2_random_demo",
)
data = loaded_run.load_table()

print(f"\nLoaded columns: {sorted(data.keys())}")
print(f"Number of rows: {len(data['k'])}")
print(f"\n{'k':>3s}  {'seed':>4s}  {'best_value':>10s}  {'elapsed_s':>9s}")
print("-" * 35)
for i in range(len(data["k"])):
    print(f"{data['k'][i]:>3d}  {data['seed'][i]:>4d}  "
          f"{data['best_value'][i]:>10.6f}  {data['elapsed_s'][i]:>9.3f}")

Available artifacts:
  bell_optimization_sweep: 1 run(s)

Loaded columns: ['best_value', 'elapsed_s', 'k', 'mask_bits', 'n_obj_evals', 'seed']
Number of rows: 6

  k  seed  best_value  elapsed_s
-----------------------------------
  1    42    2.828427      0.005
  1    43    2.828427      0.004
  2    42    2.828428      0.004
  2    43    2.828428      0.005
  3    42    2.828428      0.012
  3    43    2.828427      0.007


---
## 12. CLI Reference

For production runs (overnight, cluster, etc.) you can use the CLI tool
`scripts.bell.optimization_sweep` which wraps exactly the same functions
used above, adding argument parsing, automatic naming, and checkpoint/resume.

### Basic usage

```bash
python -m scripts.bell.optimization_sweep \
    --m-A 2 --m-B 2 --d-A 2 --d-B 2 \
    --operator chsh \
    --start-level 1 --end-level 2 \
    --method sa \
    --ks 0 1 2 3 4 \
    --seeds 42 43 44
```

### With feedback (warm-start chaining)

```bash
python -m scripts.bell.optimization_sweep \
    --m-A 2 --m-B 2 --d-A 2 --d-B 2 \
    --operator chsh \
    --start-level 1 --end-level 3 \
    --method pt \
    --k-max 10 --num-seeds 5 \
    --feedback
```

### Key flags

| Flag | Description |
|------|-------------|
| `--m-A`, `--m-B`, `--d-A`, `--d-B` | Scenario dimensions |
| `--operator` | Bell operator name (currently: `chsh`) |
| `--start-level`, `--end-level` | NPA hierarchy range |
| `--method` | `sa`, `pt`, `bo`, `rbm`, `random` |
| `--ks` | Explicit list of *k* values |
| `--k-max` | Alternative: sweep `k = 0 … k_max` |
| `--seeds`, `--num-seeds` | Explicit seed list or count |
| `--feedback` | Chain results across *k* values (warm-start) |
| `--resume` | Skip already-computed `(k, seed)` pairs |
| `--force` | Ignore existing checkpoint, start fresh |
| `--out-root` | Results directory (default: `results/`) |
| `--name` | Human-readable run name (auto-generated if omitted) |

### How CLI maps to library functions

| CLI action | Library function |
|------------|------------------|
| Parse scenario | `BellScenario(m_A, m_B, d_A, d_B)` |
| Build operator | `chsh_operator(scenario)` |
| Build basis sets | `build_bell_basis_sets(scenario, start, end)` |
| Create run dir | `ArtifactManager.create_run(...)` |
| Run sweep | `sweep_k_values(..., on_result=callback)` |
| Save results | `RunDir.save_table(...)` inside the callback |

The CLI tool uses the **exact same `on_result` callback pattern** shown
in section 10 — every result is saved atomically as soon as it's computed.